In [27]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import timedelta

# --- Base Class ---
class BaseAnalyzer(object):
    def _init__(self, file_path, output_path, target_sensors):
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.df = None
        self.df_model = None

    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"❌ فایل یافت نشد: {self.file_path}")
        self.df = pd.read_excel(self.file_path)
        if 'date' in self.df.columns:
            self.df['date'] = pd.to_datetime(self.df['date'])
        self.df_model = self.df.dropna(subset=self.target_sensors).copy()

    def preprocess(self):
        cols = []
        for sensor in self.target_sensors:
            name = f'{sensor}_smooth'
            self.df_model[name] = self.df_model[sensor].rolling(window=5, center=True).mean()
            cols.append(name)
        self.df_model = self.df_model.dropna(subset=cols).copy()
        return cols

    def save_output(self, final_df):
        try:
            os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
            final_df.to_excel(self.output_path, index=False)
            print(f"✅ خروجی در مسیر زیر ذخیره شد:\n{self.output_path}")
        except Exception as e:
            print(f"❌ خطا در ذخیره: {e}")

# --- Derived Class ---
class SmartBearingAnalyzer(BaseAnalyzer):
    def __init__(self, file_path, output_path, target_sensors, n_clusters=3):
        # مقداردهی مستقیم برای پایداری در نوت‌بوک
        self.file_path = file_path
        self.output_path = output_path
        self.target_sensors = target_sensors
        self.n_clusters = n_clusters
        self.df = None
        self.df_model = None

    def run_analysis(self):
        self.load_data()
        smooth_cols = self.preprocess()

        # ۱. استانداردسازی
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.df_model[smooth_cols])

        # ۲. کلاسترینگ (تعداد خوشه‌ها = ۳)
        model = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.df_model['Behavior_Cluster'] = model.fit_predict(scaled_data)

        # ۳. محاسبه فاصله از مرکز (دقت کنید: cluster_centers با آندراسکور)
        centers = model.cluster_centers_ 
        assigned_centers = centers[self.df_model['Behavior_Cluster']]
        self.df_model['Degradation_Index'] = np.linalg.norm(scaled_data - assigned_centers, axis=1)

        # ۴. لیبل‌گذاری
        self.df_model['Health_Status'] = self.df_model.apply(self._labeling, axis=1)

        # ۵. فیلتر ۳۰ روز آخر
        if 'date' in self.df_model.columns:
            last_dt = self.df_model['date'].max()
            final_df = self.df_model[self.df_model['date'] >= (last_dt - timedelta(days=30))].copy()
        else:
            final_df = self.df_model

        self.save_output(final_df)

    def _labeling(self, row):
        if row['Degradation_Index'] > 2.5:

            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 1.5:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

# --- CONFIG & RUN ---
CONFIG = {
    "file_path": r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx',
    "output_path": r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx',
    "sensors": ['AssetID_9343', 'AssetID_12208','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
}

# اجرا
analyzer = SmartBearingAnalyzer(CONFIG["file_path"], CONFIG["output_path"], CONFIG["sensors"])
analyzer.run_analysis()

✅ خروجی در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx
